# S07 toy — the repair loop, bounded

Companion lesson: [S07 — The repair loop](../lessons/S07-repair-loop.html).

A plant-shop copywriter: a mock generator drafts one-line product blurbs, a deterministic **scorer** rejects defective ones, and a **repair loop** regenerates with the failure as input — capped, and always ending with an honest `stop_reason`. No network, no keys; the "model" is a plain function you can read.

**How to use:** run cells in order. Each experiment has an attempt cell (write your prediction there, try it yourself) and a clearly marked SOLUTION cell after it. The gap between prediction and result is the lesson.

## The shop, the spec, the repertoire

The constraints live in a **spec**, as data: the blurb must mention the product, stay within a word cap, and avoid banned unverifiable claims. The mock generator has a fixed **repertoire** of drafts per product — its sampling distribution, made visible. Real models hide this; here you can score the whole distribution before running anything.

In [ ]:
import random
import re

DRAFTS = {
    "lavender": [
        "Amazing lavender guaranteed to thrive, the best plants for your garden",
        "Lavender for sunny borders; fragrant, hardy, loved by bees",
        "Lavender with amazing fragrance for any garden",
        "Lavender plants guaranteed to thrive in dry sunny gardens",
    ],
    "basil": [
        "Organic basil for the kitchen windowsill",
        "Basil for the kitchen windowsill",
        "Fresh organic basil, easy from seed",
    ],
}

SPEC_LAVENDER = {
    "product": "lavender",
    "must_mention": "lavender",
    "banned": ["amazing", "guaranteed", "best", "miracle"],
    "max_words": 10,
}

# The shop is not certified organic: "organic" is a banned claim.
# A template bug also *requires* the word. Nobody has noticed yet.
SPEC_BASIL = {
    "product": "basil",
    "must_mention": "organic",
    "banned": ["organic"],
    "max_words": 10,
}

STOP_REASONS = {"passed", "retries_exhausted", "policy_violation"}


## The scorer and the failure view

The scorer is deterministic code (S02's deterministic tier): it returns pass/fail plus **structured failures** — which check, which span, which constraint. `failure_view` renders those failures into the *only* new information a retry can carry. That text is prompt content, and it is designed like an interface: name the check, quote the span, state the constraint, change nothing else.

In [ ]:
def tokens(text):
    return re.findall(r"[a-z0-9']+", text.lower())

def score(draft, spec):
    """Deterministic tier: structure and policy, not quality."""
    words = tokens(draft)
    failures = []
    for w in spec["banned"]:
        if w in words:
            failures.append({"check": "banned", "span": w,
                             "constraint": f"banned word: '{w}' (unverifiable claim) — remove or replace it"})
    if len(words) > spec["max_words"]:
        failures.append({"check": "length", "span": None,
                         "constraint": f"too long: {len(words)} words (max {spec['max_words']}) — shorten"})
    if spec["must_mention"] not in words:
        failures.append({"check": "mention", "span": None,
                         "constraint": f"must mention '{spec['must_mention']}'"})
    return {"passed": not failures, "failures": failures}

def failure_view(failures, attempt):
    """The retry's only new information. Concise, localized, actionable."""
    lines = [f"[validator feedback] draft {attempt} rejected — fix exactly these, change nothing else:"]
    lines += [f"- {f['constraint']}" for f in failures]
    return "\n".join(lines)


## The mock generator

A rule-based stand-in for a chat-completions call, API-shaped like S01's. Three designed behaviors, each mirroring a documented model behavior:

- **no feedback in context** → resample from the repertoire (temperature: variance without direction);
- **a failure view in context** → fix the named defects (models comply with precise, localized instructions);
- **failed drafts visible in context** → *imitation*: it cannot let go of a banned word it can see, even when the feedback names it. This exaggerates a real tendency — models repeat salient text — so you can watch it happen.

In [ ]:
def _respond(draft):
    msg = {"role": "assistant", "content": draft}
    return {"choices": [{"message": msg}], "usage": {"total_tokens": 12 + len(draft.split())}}

def banned_in(draft, spec):
    return {w for w in spec["banned"] if w in tokens(draft)}

def mock_generator(messages, spec, rng):
    pool = DRAFTS[spec["product"]]
    feedback = [m["content"] for m in messages if "[validator feedback]" in m["content"]]
    if not feedback:
        return _respond(rng.choice(pool))            # no new information -> resample
    last_fb = feedback[-1]
    visible = [m["content"] for m in messages if m["role"] == "assistant"]
    anchored = set().union(*(banned_in(d, spec) for d in visible)) if visible else set()
    candidates = pool
    if "too long" in last_fb:
        candidates = [d for d in candidates if len(tokens(d)) <= spec["max_words"]]
    if "must mention" in last_fb:
        candidates = [d for d in candidates if spec["must_mention"] in tokens(d)]
    if anchored:
        sticky = [d for d in candidates if banned_in(d, spec) & anchored]
        if sticky:
            return _respond(sticky[0])               # imitation beats instruction
    named = {w for w in spec["banned"] if f"'{w}'" in last_fb}
    if named:
        fixed = [d for d in candidates if not banned_in(d, spec) & named]
        if fixed:
            candidates = fixed
    return _respond(candidates[0] if candidates else pool[0])


## The repair loop

Score → on failure, regenerate with the failure view → re-score, at most `cap` times. Three retry-context **modes** differ in exactly one thing — what the retry gets to see:

| mode | retry context |
|---|---|
| `naive` | the identical brief (no new information) |
| `full_history` | brief + every failed draft + feedback |
| `curated` | brief + failure view (failed drafts dropped) |

The loop returns a **run record**: `stop_reason`, the accepted draft (or `None`), and every attempt. A policy-listed product never enters the loop at all — that defect is in the request, not the draft. No run ends ambiguously.

In [ ]:
def brief_for(spec):
    return (f"Write a one-line product blurb for our {spec['product']}. "
            f"Rules: mention '{spec['must_mention']}'; at most {spec['max_words']} words; "
            "no unverifiable claims.")

def repair_loop(spec, mode, cap=3, seed=7):
    rng = random.Random(seed)
    if spec["product"] in spec.get("do_not_promote", ()):
        return {"stop_reason": "policy_violation", "draft": None, "attempts": []}
    brief = brief_for(spec)
    messages = [{"role": "user", "content": brief}]
    attempts = []
    for n in range(1, cap + 1):
        body = mock_generator(messages, spec, rng)
        draft = body["choices"][0]["message"]["content"]
        result = score(draft, spec)
        attempts.append({"n": n, "draft": draft, "failures": result["failures"]})
        if result["passed"]:
            return {"stop_reason": "passed", "draft": draft, "attempts": attempts}
        view = failure_view(result["failures"], n)
        if mode == "naive":
            pass                      # identical context: resampling, not repair
        elif mode == "curated":
            messages = [{"role": "user", "content": brief + "\n\n" + view}]
        elif mode == "full_history":
            messages = messages + [{"role": "assistant", "content": draft},
                                   {"role": "user", "content": view}]
        else:
            raise ValueError(mode)
    return {"stop_reason": "retries_exhausted", "draft": None, "attempts": attempts}

def show(run):
    for a in run["attempts"]:
        fails = "; ".join(f["constraint"] for f in a["failures"]) or "clean"
        print(f"  attempt {a['n']}: {a['draft']!r}\n           -> {fails}")
    print(f"  stop_reason={run['stop_reason']}  draft={run['draft']!r}")


## Experiment 1 — the repertoire, scored

**Predict first:** which of the four lavender drafts passes? Which check kills each of the others? Write it down, then run the SOLUTION cell.

In [ ]:
# YOUR ATTEMPT — Exercise 1
# Prediction: draft ___ passes; draft ___ fails check ___, ...
# Try it: call score() on each draft in DRAFTS["lavender"] and inspect the failures.
# Then run the SOLUTION cell below.


In [ ]:
# ===== SOLUTION — Exercise 1 (run after your attempt) =====
for d in DRAFTS["lavender"]:
    r = score(d, SPEC_LAVENDER)
    verdict = "PASS" if r["passed"] else "FAIL " + ", ".join(f["check"] for f in r["failures"])
    print(f"{verdict:<28} {d!r}")

print("\nOne draft in four passes. That 25% is the mock's per-draft pass rate p —")
print("every naive retry below is a draw from this distribution.")


## Experiment 2 — naive retry is resampling

The loop retries with the *identical* brief — no new information. **Predict first:** which drafts appear, and what is the stop reason? Then: across 30 seeds, how often does resampling luck through, and what does that number have to do with p = 0.25?

In [ ]:
# YOUR ATTEMPT — Exercise 2
# Prediction: stop_reason = ___; naive pass rate over 30 seeds ≈ ___
# Try it: run repair_loop(SPEC_LAVENDER, "naive", seed=<pick one>) and show() it.
# Then run the SOLUTION cell below.


In [ ]:
# ===== SOLUTION — Exercise 2 (run after your attempt) =====
show(repair_loop(SPEC_LAVENDER, "naive", seed=6))

wins = sum(repair_loop(SPEC_LAVENDER, "naive", seed=s)["stop_reason"] == "passed"
           for s in range(30))
p = 1 / 4                       # from Experiment 1: one draft in four passes
print(f"\nnaive retry passed in {wins}/30 seeds")
print(f"lottery theory (independent draws): 1-(1-p)^cap = {1 - (1 - p) ** 3:.3f}")
print("The drafts change every attempt; the failure class doesn't. Variance is not direction.")


## Experiment 3 — the curated failure view

Same generator, same cap — but the retry carries the failure view, and the failed draft is dropped from context. **Predict first:** which attempt passes, and with which draft? After running, read the exact feedback text that was sent — would *you* know what to fix from it?

In [ ]:
# YOUR ATTEMPT — Exercise 3
# Prediction: attempt ___ passes with draft ___
# Try it: run repair_loop(SPEC_LAVENDER, "curated", seed=5) and show() it.
# Then run the SOLUTION cell below.


In [ ]:
# ===== SOLUTION — Exercise 3 (run after your attempt) =====
run = repair_loop(SPEC_LAVENDER, "curated", seed=5)
show(run)

# The exact message the generator saw before its passing attempt:
first = mock_generator([{"role": "user", "content": brief_for(SPEC_LAVENDER)}],
                       SPEC_LAVENDER, random.Random(5))
first_draft = first["choices"][0]["message"]["content"]
print("\n--- the failure view sent before attempt 2 ---")
print(failure_view(score(first_draft, SPEC_LAVENDER)["failures"], 1))
print("One line, one span, one constraint. That is the whole mechanism.")


## Experiment 4 — what context does the retry get?

`full_history` mode keeps every failed draft in context "for transparency." The mock's imitation rule now applies: a banned word it can *see* is sticky, even when the feedback names it. **Predict first:** what does attempt 2 fix, and what does it keep? What is the stop reason?

In [ ]:
# YOUR ATTEMPT — Exercise 4
# Prediction: attempt 2 keeps ___ ; stop_reason = ___
# Try it: run repair_loop(SPEC_LAVENDER, "full_history", seed=2) and show() it.
# Then run the SOLUTION cell below.


In [ ]:
# ===== SOLUTION — Exercise 4 (run after your attempt) =====
show(repair_loop(SPEC_LAVENDER, "full_history", seed=2))

print("\nAttempt 2 fixed the length and kept a banned phrase it could see.")
print("Attempt 3 went back to the first failure. The loop is orbiting its own")
print("rejections — visible failures anchor. This is why 'what context does a")
print("retry get' is a recorded decision, not a default.")


## Experiment 5 — the unfixable defect and the honest stop

`SPEC_BASIL` is contradictory: the blurb *must mention* a word that is *banned*. No draft can pass — the mock doesn't know that, and neither does the loop. **Predict first:** the stop reason, and what (if anything) ships. Then answer in a comment: which failure classes must never enter the loop at all?

In [ ]:
# YOUR ATTEMPT — Exercise 5
# Prediction: stop_reason = ___; shipped draft = ___
# Classes that must never enter the loop: ___
# Try it: run repair_loop(SPEC_BASIL, "curated", seed=1) and show() it.
# Then run the SOLUTION cell below.


In [ ]:
# ===== SOLUTION — Exercise 5 (run after your attempt) =====
run = repair_loop(SPEC_BASIL, "curated", seed=1)
show(run)
assert run["stop_reason"] == "retries_exhausted" and run["draft"] is None
print("\nThe failure view ping-pongs: fix the banned word, lose the required one,")
print("fix that, get the banned word back. The cap converts the contradiction into")
print("an honest stop — and the attempt log is the diagnosis surface for a human.")

# The policy class never enters the loop:
blocked = repair_loop(dict(SPEC_BASIL, do_not_promote=("basil",)), "curated", seed=1)
show(blocked)
assert blocked["attempts"] == []

# The invariant, executable: across every mode, every run ends with a known reason.
for mode in ("naive", "curated", "full_history"):
    for seed in range(10):
        r = repair_loop(SPEC_LAVENDER, mode, seed=seed)
        assert r["stop_reason"] in STOP_REASONS, r
        assert (r["draft"] is None) == (r["stop_reason"] != "passed"), r
print("\nNo run ends ambiguously; nothing ships without passing the scorer.")


## What transfers to the real build

- `score()` → the deterministic checks each turn is scored against (the S02 tier, applied per turn and per session).
- `failure_view()` → the curated context a regeneration attempt receives. Naive / full-history / curated is the decision you now have to make and record — you have watched all three fail differently.
- `cap=3` → the attempt budget; `stop_reason` → the terminal state, printed and stored. The real build's set is wider (budget, turn cap, safety handoff) but the rule is the toy's rule: no session ends ambiguously.
- `do_not_promote` → policy violations are defects in the *request*: they stop the run, they never enter the loop, and regeneration never grows the approved scope.
- What the toy doesn't have: a judged tier for semantic defects (uncalibrated until S12), per-attempt cost charged against a budget (S11), and any record of the failed drafts once the run ends — which is exactly the hole S08 fills with traces and replay.

Now do the real build in your own project. You type it.